# Calculating adequate sample size across clinical scenarios with computational implementation in r and python

**Authors:** Renato Carneiro de Freitas Chaves, Tiago Mendonça dos Santos, Thiago Domingos Corrêa


## 1) Purpose
This document presents a practical document for sample-size calculations in common clinical and epidemiological scenarios.

The examples use a small number of parameters that can be edited directly by the researcher.



## 2) Required Libraries

This notebook uses standard scientific Python libraries.

In [ ]:
import math
from statistics import NormalDist

import pandas as pd
import numpy as np

## 2. Data Import

This guide assumes that an Excel file named **`data.xlsx`** is available in the same folder as this notebook, with a worksheet named **`Data`**.

Expected variables:

- `group`: 0 = control, 1 = intervention
- `outcome`: 0 = no event, 1 = event

The descriptive estimates derived from the dataset are optional and should not replace clinically justified assumptions.

In [ ]:
data = pd.read_excel("data.xlsx", sheet_name="Data")

data.head()

## 3. User-Defined Parameters

Edit this section according to the study design, the target population, and the expected clinical effect. The values below are illustrative.

In [ ]:
# General parameters
alpha = 0.05
power = 0.80
attrition = 0.10

# Dataset variables
group_var = "group"
outcome_var = "outcome"

# Precision and effect-size assumptions
precision_prevalence = 0.05
precision_mean = 3
minimal_difference_mean = 5

# Default assumptions
p_control_default = 0.25
p_intervention_default = 0.15
sd_default = 10

# Case-control assumptions
odds_ratio = 2.0
p_exposed_controls = 0.20

# Diagnostic-accuracy assumptions
sensitivity = 0.85
specificity = 0.90
precision_sensitivity = 0.07
precision_specificity = 0.05
disease_prevalence = 0.20

## 4. Descriptive estimates from the dataset

These estimates are used as data-informed examples. In a formal protocol, assumptions should be clinically and methodologically justified rather than selected only from the observed dataset.

In [ ]:
overall_event_risk = data[outcome_var].mean()

risk_by_group = (
    data
    .groupby(group_var, as_index=False)
    .agg(
        n=(outcome_var, "size"),
        event_risk=(outcome_var, "mean")
    )
)

risk_by_group

In [ ]:
risk_table = risk_by_group.set_index(group_var)["event_risk"]

risk_control = risk_table.get(0, p_control_default)
risk_intervention = risk_table.get(1, p_intervention_default)
continuous_sd = sd_default

assumptions = pd.DataFrame({
    "parameter": [
        "overall_event_risk",
        "risk_control",
        "risk_intervention",
        "continuous_sd",
        "alpha",
        "power",
        "attrition"
    ],
    "value": [
        overall_event_risk,
        risk_control,
        risk_intervention,
        continuous_sd,
        alpha,
        power,
        attrition
    ]
})

assumptions

## 5. Statistical constants

For two-sided tests, the alpha quantile is calculated as `1 - alpha / 2`. The power quantile is calculated as `power`.

In [ ]:
normal = NormalDist()

z_alpha = normal.inv_cdf(1 - alpha / 2)
z_power = normal.inv_cdf(power)

constants = pd.DataFrame({
    "parameter": ["Z for alpha", "Z for power"],
    "value": [z_alpha, z_power]
})

constants

## 6. Scenario 1: Prevalence survey

For a single proportion estimated with absolute precision:

$$
n = \frac{Z_{1-\alpha/2}^{2}p(1-p)}{d^2}
$$

In [ ]:
p = overall_event_risk
d = precision_prevalence

n_prevalence = z_alpha**2 * p * (1 - p) / d**2
n_prevalence_final = math.ceil(n_prevalence / (1 - attrition))

prevalence_result = pd.DataFrame({
    "scenario": ["Prevalence survey"],
    "expected_prevalence": [p],
    "precision": [d],
    "sample_size_before_attrition": [math.ceil(n_prevalence)],
    "final_sample_size": [n_prevalence_final]
})

prevalence_result

## 7. Scenario 2: estimating a single mean

For a single mean estimated with absolute precision:

$$
n = \frac{Z_{1-\alpha/2}^{2}\sigma^2}{d^2}
$$

In [ ]:
sigma = continuous_sd
d = precision_mean

n_single_mean = z_alpha**2 * sigma**2 / d**2
n_single_mean_final = math.ceil(n_single_mean / (1 - attrition))

single_mean_result = pd.DataFrame({
    "scenario": ["Single mean"],
    "expected_sd": [sigma],
    "precision": [d],
    "sample_size_before_attrition": [math.ceil(n_single_mean)],
    "final_sample_size": [n_single_mean_final]
})

single_mean_result

## 8. Scenario 3: Two independent proportions

For equal allocation and a two-sided superiority test:

$$
n_{per\ group} = \frac{\left[Z_{1-\alpha/2}\sqrt{2\bar{p}(1-\bar{p})} + Z_{1-\beta}\sqrt{p_1(1-p_1)+p_2(1-p_2)}\right]^2}{(p_1-p_2)^2}
$$

In [ ]:
p1 = risk_control
p2 = risk_intervention
p_bar = (p1 + p2) / 2
risk_difference = abs(p1 - p2)

n_two_prop_group = (
    z_alpha * math.sqrt(2 * p_bar * (1 - p_bar))
    + z_power * math.sqrt(p1 * (1 - p1) + p2 * (1 - p2))
)**2 / risk_difference**2

n_two_prop_group_final = math.ceil(n_two_prop_group / (1 - attrition))

two_prop_result = pd.DataFrame({
    "scenario": ["Two independent proportions"],
    "control_event_risk": [p1],
    "intervention_event_risk": [p2],
    "absolute_risk_difference": [p2 - p1],
    "sample_size_per_group_before_attrition": [math.ceil(n_two_prop_group)],
    "final_sample_size_per_group": [n_two_prop_group_final],
    "final_total_sample_size": [2 * n_two_prop_group_final]
})

two_prop_result

## 9. Scenario 4: Two independent means

For two independent groups with equal allocation:

$$
n_{per\ group} = \frac{2\sigma^2\left(Z_{1-\alpha/2}+Z_{1-\beta}\right)^2}{\Delta^2}
$$

In [ ]:
sigma = continuous_sd
delta = minimal_difference_mean

n_two_means_group = 2 * sigma**2 * (z_alpha + z_power)**2 / delta**2
n_two_means_group_final = math.ceil(n_two_means_group / (1 - attrition))

two_means_result = pd.DataFrame({
    "scenario": ["Two independent means"],
    "expected_sd": [sigma],
    "clinically_meaningful_difference": [delta],
    "sample_size_per_group_before_attrition": [math.ceil(n_two_means_group)],
    "final_sample_size_per_group": [n_two_means_group_final],
    "final_total_sample_size": [2 * n_two_means_group_final]
})

two_means_result

## 10. Scenario 5: Unmatched case-control study

The expected exposure prevalence among cases can be approximated from the exposure prevalence among controls and the odds ratio:

$$
p_1 = \frac{OR \times p_0}{1 - p_0 + OR \times p_0}
$$

In [ ]:
p0 = p_exposed_controls
p1 = odds_ratio * p0 / (1 - p0 + odds_ratio * p0)
p_bar = (p0 + p1) / 2
exposure_difference = abs(p1 - p0)

n_case_control_group = (
    z_alpha * math.sqrt(2 * p_bar * (1 - p_bar))
    + z_power * math.sqrt(p0 * (1 - p0) + p1 * (1 - p1))
)**2 / exposure_difference**2

n_case_control_group_final = math.ceil(n_case_control_group / (1 - attrition))

case_control_result = pd.DataFrame({
    "scenario": ["Unmatched case-control study"],
    "exposure_prevalence_controls": [p0],
    "expected_odds_ratio": [odds_ratio],
    "exposure_prevalence_cases": [p1],
    "sample_size_per_group_before_attrition": [math.ceil(n_case_control_group)],
    "final_sample_size_per_group": [n_case_control_group_final],
    "final_total_sample_size": [2 * n_case_control_group_final]
})

case_control_result

## 11. Scenario 6: Diagnostic-accuracy study

For sensitivity:

$$
n_{diseased} = \frac{Z_{1-\alpha/2}^{2}Se(1-Se)}{d_{Se}^{2}}
$$

For specificity:

$$
n_{non-diseased} = \frac{Z_{1-\alpha/2}^{2}Sp(1-Sp)}{d_{Sp}^{2}}
$$

In [ ]:
n_diseased = z_alpha**2 * sensitivity * (1 - sensitivity) / precision_sensitivity**2
n_nondiseased = z_alpha**2 * specificity * (1 - specificity) / precision_specificity**2

total_for_sensitivity = math.ceil(n_diseased / disease_prevalence)
total_for_specificity = math.ceil(n_nondiseased / (1 - disease_prevalence))
total_diagnostic = max(total_for_sensitivity, total_for_specificity)
total_diagnostic_final = math.ceil(total_diagnostic / (1 - attrition))

diagnostic_result = pd.DataFrame({
    "scenario": ["Diagnostic-accuracy study"],
    "expected_sensitivity": [sensitivity],
    "expected_specificity": [specificity],
    "disease_prevalence": [disease_prevalence],
    "diseased_required": [math.ceil(n_diseased)],
    "nondiseased_required": [math.ceil(n_nondiseased)],
    "total_before_attrition": [total_diagnostic],
    "final_total_sample_size": [total_diagnostic_final]
})

diagnostic_result

## 12. Final summary

The table below consolidates the final sample-size estimates across scenarios. Values should be reviewed in light of the study question, feasibility, ethical considerations, and the assumptions used in each calculation.

In [ ]:
final_summary_table = pd.DataFrame({
    "Scenario": [
        "Prevalence survey",
        "Single mean",
        "Two independent proportions",
        "Two independent means",
        "Unmatched case-control study",
        "Diagnostic-accuracy study"
    ],
    "Sample_size": [
        prevalence_result.loc[0, "final_sample_size"],
        single_mean_result.loc[0, "final_sample_size"],
        two_prop_result.loc[0, "final_total_sample_size"],
        two_means_result.loc[0, "final_total_sample_size"],
        case_control_result.loc[0, "final_total_sample_size"],
        diagnostic_result.loc[0, "final_total_sample_size"]
    ],
    "Note": [
        "Total participants",
        "Total participants",
        "Total participants",
        "Total participants",
        "Cases plus controls",
        "Total participants"
    ]
})

final_summary_table